In [15]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
import json
# loading the dataset
malgenome_df = pd.read_csv('malgenome-215_shuffled.csv')

In [16]:
malgenome_df.head(10)

,transact,bindService,onServiceConnected,ServiceConnection,android.os.Binder,READ_SMS,attachInterface,WRITE_SMS,TelephonyManager.getSubscriberId,Ljava.lang.Class.getCanonicalName,...,Ljava.lang.Object.getClass,SET_ORIENTATION,DEVICE_POWER,EXPAND_STATUS_BAR,GET_TASKS,GLOBAL_SEARCH,GET_PACKAGE_SIZE,SET_PREFERRED_APPLICATIONS,android.intent.action.PACKAGE_CHANGED,class
0,0,0,0,0,0,1,0,1,1,0,...,1,0,0,0,0,0,0,0,0,S
1,0,0,0,0,0,1,0,1,1,0,...,1,0,0,0,0,0,0,0,0,S
2,0,0,0,0,0,1,0,1,1,0,...,1,0,0,0,1,0,0,0,0,S
3,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,S
4,1,1,1,1,1,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,B
5,0,0,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,B
6,1,1,1,1,1,0,1,0,0,1,...,1,0,0,0,0,1,0,0,0,B
7,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,B
8,0,0,0,0,0,1,0,1,1,0,...,1,0,0,0,0,0,0,0,0,S
9,1,1,1,1,1,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,B


In [17]:
# Label mapping: 'B' → 0, 'S' → 1
label_map = {'B': 0, 'S': 1}
y = malgenome_df["class"].map(label_map)

In [18]:
# Drop any rows where label could not be mapped
valid_idx = y.dropna().index
malgenome = malgenome_df.loc[valid_idx]
y = y.loc[valid_idx]

In [19]:
X = malgenome.drop(columns=["class"])

In [20]:
non_numeric_cols = X.select_dtypes(include=["object"]).columns
for col in non_numeric_cols:
    # dropping the rows with non-numeric vbalues
    X = X[X[col].apply(lambda x: str(x).isdigit())]
    y = y.loc[X.index] #keep labels in sync
    X[col] = X[col].astype(int)

# confirming that all columns are numeric
X = X.fillna(0).astype(int)

In [21]:
X.dtypes.value_counts()

int64    215
Name: count, dtype: int64

In [ ]:
# Check the number of rows and columns in label DataFrame
y.value_counts()


class
0    2539
1    1260
Name: count, dtype: int64

In [ ]:
# Splitting the dataset into training and testing sets
# Stratified to maintain the class distribution
# 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# The stratify argument in train_test_split ensures that the target variable (y) is evenly distributed in the training and testing sets.
#  This means that the proportion of each label in the test set will match the proportion of each label in the entire dataset.
# For example, if your dataset has 80% labels with value 0 and 20% labels with value 1, the stratify argument will try to maintain this balance in the training and testing sets.

In [25]:
# feature importance mutual information(MI)
mi_scores = mutual_info_classif(X_train, y_train, discrete_features=True, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns)
top_features_mi = mi_series.sort_values(ascending=False).head(100)

In [27]:
# feature importance (Random Forest)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train,y_train)
# get the feature importances
# The feature_importances_ attribute of the RandomForestClassifier object gives the importance of each feature in the model.
rf_importances = pd.Series(rf.feature_importances_, index = X.columns)
# Sort the importances in descending order
top_features_rf = rf_importances.sort_values(ascending=False).head(100)

In [28]:
# saving the union of the two feature importance lists
top_features = list(set(top_features_mi.index).union(set(top_features_rf.index)))
print(f"Selected {len(top_features)}  features from MalGenome dataset")

Selected 122  features from MalGenome dataset


In [29]:

with open("malgenome_top_features.json", "w") as f:
    json.dump(top_features, f)
